#Setup

In [ ]:
from google.colab import drive
# Run from the repository root.

# Run from the repository root.

In [ ]:
from google.colab import userdata
hf_token = userdata.get('hugging_face')
openai_token = userdata.get('openai')

In [ ]:
#import dataset
import pandas as pd

df = pd.read_csv('data/raw/CS263_dataset.csv')
df.head()

#DeBERTa 2 Class + Threshold
We chose DeBERTa as one of our base models because its a model that is optimized for Natural Language Inference, trained to handle entailment-style reasoning.

**Design**

Have DeBERTa predict between "acceptable" and "not acceptable", if the confidence is below a threshold, return "context-dependent" instead.

In [ ]:
import re
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score

# 1. Load dataset
df = pd.read_csv('data/raw/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load DeBERTa MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="microsoft/deberta-large-mnli",
    token=hf_token
)

candidate_labels = ["acceptable behavior", "unacceptable behavior"]

def predict_acceptability(situation, threshold=0.65):
    result = classifier(
        situation,
        candidate_labels=[
            "acceptable behavior",
            "not acceptable behavior"
        ],
        hypothesis_template="This behavior is {}."
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < threshold:
        return "context-dependent"

    if top_label == "acceptable behavior":
        return "acceptable"
    else:
        return "not acceptable"

# 5. Predict
predictions = []

for _, row in df.iterrows():

    pred_label = predict_acceptability(row["situation"])

    predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

# 6. Create result DataFrame
results_df = pd.DataFrame(predictions)

results_df.head()
results_df.to_csv("deberta_predictions.csv", index=False)

In [ ]:
!pip install openai

#ChatGPT

In [ ]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt(situation):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Do not use cultural context unless it is explicitly included in the situation.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()
gpt_results_df.to_csv("gpt_predictions.csv", index=False)

In [ ]:
SYSTEM_PROMPT = """
You are a careful social acceptability classifier.

Classify the interaction into exactly one label:
- acceptable
- not acceptable
- context-dependent

You are given:
1. interaction: the user/LLM exchange
2. culture: the relevant cultural or social norm
3. cultural_context: explanation of that norm

Use the provided culture and cultural_context when judging acceptability.
Do not invent additional cultural assumptions beyond what is provided.

Label definitions:
- acceptable: The behavior is generally socially appropriate under the given cultural context.
- not acceptable: The behavior clearly violates the given cultural/social norm, or is rude, harmful, invasive, disrespectful, unsafe, or unfair.
- context-dependent: The acceptability depends on missing details such as tone, closeness, consent, setting, urgency, frequency, age, relationship, or power dynamics.

Decision rules:
1. If privacy, consent, harassment, dishonesty, safety, or discrimination is involved, be stricter.
2. If the provided cultural context says expectations vary or depend on circumstances, prefer context-dependent.
3. If the behavior is a respectful boundary and the context supports autonomy/privacy, prefer acceptable.
4. If the behavior publicly embarrasses, invades privacy, violates trust, or disrespects a strong norm, prefer not acceptable.
5. Think internally before deciding, but output only valid JSON.

Return exactly this JSON format:
{"label": "acceptable"}
"""

def updated_predict_acceptability_gpt(row):
    user_prompt = f"""
Interaction:
{row["interaction"]}

Culture:
{row["culture"]}

Cultural context:
{row["cultural_context"]}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
from tqdm import tqdm

In [ ]:
# -------------------------
# 4. Run predictions
# -------------------------
gpt_predictions = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_label = updated_predict_acceptability_gpt(row)

    gpt_predictions.append({
        "id": row["id"],
        "category": row["category"],
        "interaction": row["interaction"],
        "culture": row["culture"],
        "cultural_context": row["cultural_context"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()

In [ ]:
# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")

In [ ]:
FEW_SHOT = """
Examples:

Interaction:
User: I looked through my partner’s phone because I felt suspicious.
LLM: That raises privacy and trust concerns.
Culture:
US, relationship-privacy norm
Cultural context:
Privacy and trust are important in relationships.
Output:
{"label": "not acceptable"}

Interaction:
User: I skipped a family dinner because I wanted alone time.
LLM: That’s understandable. Did you explain it?
Culture:
US, personal-boundary norm
Cultural context:
Personal time and emotional boundaries are generally accepted, especially if communicated respectfully.
Output:
{"label": "acceptable"}

Interaction:
User: I didn’t help my sibling financially.
LLM: That depends on your situation.
Culture:
India, extended-family norm
Cultural context:
Family support may be expected, but obligation depends on resources, relationship, and circumstances.
Output:
{"label": "context-dependent"}
"""

SYSTEM_PROMPT = SYSTEM_PROMPT + "\n\n" + FEW_SHOT

In [ ]:
# -------------------------
# 4. Run predictions
# -------------------------
gpt_predictions = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_label = updated_predict_acceptability_gpt(row)

    gpt_predictions.append({
        "id": row["id"],
        "category": row["category"],
        "interaction": row["interaction"],
        "culture": row["culture"],
        "cultural_context": row["cultural_context"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()

In [ ]:
# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")